In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F
spark.conf.set("spark.sql.session.timeZone", "UTC")
SIGNALS = ["gps", "engine", "fuel", "temperature", "device_status"]

StatementMeta(, 27d80fed-0033-4573-8774-0ac4a9f0c0f3, 3, Finished, Available, Finished, False)

In [2]:
for sch in ["ops", "ref"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {sch}")

spark.sql("""CREATE TABLE IF NOT EXISTS ops.ops_watermark
  (signal STRING, watermark_utc STRING, updated_at TIMESTAMP) USING DELTA""")

spark.sql("""CREATE TABLE IF NOT EXISTS ops.dlq_events (
  dlq_id STRING, batch_id STRING, stage STRING, signal STRING,
  error_code STRING, error_message STRING, event_id STRING,
  raw_value STRING, source_type STRING, source_file STRING, file_line INT,
  bronze_ingestion_time TIMESTAMP, is_retryable BOOLEAN, created_at TIMESTAMP
) USING DELTA""")

spark.sql("""CREATE TABLE IF NOT EXISTS ops.ops_ingestion_log (
  log_id STRING, batch_id STRING, job_name STRING, signal STRING, stage STRING,
  source_file STRING, window_start_utc STRING, window_end_utc STRING,
  rows_bronze BIGINT, rows_silver BIGINT, rows_dlq BIGINT, rows_duplicates_removed BIGINT,
  status STRING, error_message STRING, started_at TIMESTAMP, finished_at TIMESTAMP
) USING DELTA""")

if spark.table("ops.ops_watermark").count() == 0:
    (spark.createDataFrame([(s, "1900-01-01T00:00:00Z") for s in SIGNALS],
                           "signal string, watermark_utc string")
          .withColumn("updated_at", F.current_timestamp())
          .write.mode("append").saveAsTable("ops.ops_watermark"))

StatementMeta(, 888121a3-0cbc-40dd-beee-53a4ed4e682d, 4, Finished, Available, Finished, False)

In [3]:
for f in mssparkutils.fs.ls("Files/reference"):
    if not f.name.endswith(".csv") or f.name.startswith("vehicles_"):
        continue
    df = spark.read.option("header", True).option("inferSchema", True).csv(f.path)
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"ref.ref_{f.name[:-4]}")

vdf = (spark.read.option("header", True).csv("Files/reference/vehicles_*.csv")
         .withColumn("source_file", F.element_at(F.split(F.input_file_name(), "/"), -1)))
vdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ref.ref_vehicles_raw")

StatementMeta(, 888121a3-0cbc-40dd-beee-53a4ed4e682d, 5, Finished, Available, Finished, False)

In [4]:
for t in ["ref.ref_drivers","ref.ref_routes","ref.ref_route_checkpoints","ref.ref_cargo_types",
          "ref.ref_trips","ref.ref_fuel_prices","ref.ref_business_rules","ref.ref_vehicles_raw",
          "ops.ops_watermark","ops.dlq_events","ops.ops_ingestion_log"]:
    print(t, spark.table(t).count())
mssparkutils.notebook.exit("done")

StatementMeta(, 888121a3-0cbc-40dd-beee-53a4ed4e682d, 6, Finished, Available, Finished, False)

ref.ref_drivers 13
ref.ref_routes 3
ref.ref_route_checkpoints 59
ref.ref_cargo_types 4
ref.ref_trips 21
ref.ref_fuel_prices 2
ref.ref_business_rules 32
ref.ref_vehicles_raw 23
ops.ops_watermark 5
ops.dlq_events 0
ops.ops_ingestion_log 0


In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("SHOW SCHEMAS").show()

StatementMeta(, 27d80fed-0033-4573-8774-0ac4a9f0c0f3, 4, Finished, Available, Finished, False)

+--------------------+
|           namespace|
+--------------------+
|SIC8-Fleet-Telema...|
|SIC8-Fleet-Telema...|
|SIC8-Fleet-Telema...|
|SIC8-Fleet-Telema...|
|SIC8-Fleet-Telema...|
+--------------------+



In [ ]:
# 1. Check the Bronze tables
bronze_tables = [
    "bronze.bronze_gps_raw",
    "bronze.bronze_engine_raw",
    "bronze.bronze_fuel_raw",
    "bronze.bronze_temperature_raw",
    "bronze.bronze_device_status_raw"
]

for table in bronze_tables:
    try:
        count = spark.table(table).count()
        print(f"{table}: {count} rows")
    except Exception as e:
        print(f"{table}: ERROR - {str(e)[:200]}")

# 2. Check the Batch Job 1 watermark
spark.sql("""
    SELECT *
    FROM ops.ops_watermark
    ORDER BY signal
""").show(truncate=False)

StatementMeta(, 27d80fed-0033-4573-8774-0ac4a9f0c0f3, 5, Finished, Available, Finished, False)

bronze.bronze_gps_raw: 30895 rows


bronze.bronze_engine_raw: 30926 rows


bronze.bronze_fuel_raw: 5176 rows


bronze.bronze_temperature_raw: 7197 rows
bronze.bronze_device_status_raw: 0 rows


+-------------+--------------------+--------------------------+
|signal       |watermark_utc       |updated_at                |
+-------------+--------------------+--------------------------+
|device_status|1900-01-01T00:00:00Z|2026-10-26 05:41:53.598443|
|engine       |1900-01-01T00:00:00Z|2026-10-26 05:41:53.598443|
|fuel         |1900-01-01T00:00:00Z|2026-10-26 05:41:53.598443|
|gps          |1900-01-01T00:00:00Z|2026-10-26 05:41:53.598443|
|temperature  |1900-01-01T00:00:00Z|2026-10-26 05:41:53.598443|
+-------------+--------------------+--------------------------+



In [1]:

tables_to_check = [
    "bronze.bronze_gps_raw",
    "bronze.bronze_engine_raw",
    "bronze.bronze_fuel_raw",
    "bronze.bronze_temperature_raw",
    "ref.ref_trips",
    "ops.dlq_events",
    "ops.ops_ingestion_log"
]

for table in tables_to_check:
    print(f"\n{'=' * 50}\nTABLE: {table}")
    try:
        spark.table(table).printSchema()
        spark.table(table).show(2, truncate=False)
    except Exception as e:
        print(f"ERROR: {str(e)[:250]}")


StatementMeta(, 0abd5e77-089a-4a6a-805a-accb64f53c58, 3, Finished, Available, Finished, False)


TABLE: bronze.bronze_gps_raw
root
 |-- raw_value: string (nullable = true)
 |-- source_stream: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- source_file: string (nullable = true)
 |-- file_line: integer (nullable = true)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-----------+-----------------------+--------------+---------------+------------+-----------+---------+
|raw_value                                                                                                                 

In [ ]:
# Release cached data and clear Spark cache
spark.catalog.clearCache()